In [5]:
!pip install scikit-learn pandas

In [5]:
import pandas as pd
import sklearn
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version: 3.0.3
Scikit-learn version: 1.9.0


In [7]:
import os
print("Current working directory:", os.getcwd())
print("\nOne level up:", os.listdir('..'))

Current working directory: c:\Users\Bisma\Documents\GitHub\FORUM_POSTS.CSV\forum_posts.csv\week 3\week 4

One level up: ['clean_dataset.csv', 'data_cleaning.ipynb', 'week 4']


In [9]:
df = pd.read_csv('../clean_dataset.csv')
df.head()

,content,clean_content
0,“The world as we have created it is a process ...,“the world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t...","“it is our choices, harry, that show what we t..."
2,“There are only two ways to live your life. On...,“there are only two ways to live your life. on...
3,"“The person, be it gentleman or lady, who has ...","“the person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and...","“imperfection is beauty, madness is genius and..."


In [14]:
!pip install textblob

   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/625.0 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/625.0 kB ? eta -:--:--
   ---------------------------------------- 625.0/625.0 kB 1.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------ --------- 1.3/1.7 MB 7.0 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 6.3 MB/s  0:00:00

   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [nltk]
   ---------------------------------------- 0/2 [

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [15]:
from textblob import TextBlob

def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0.1:
        return 'positive'
    elif polarity < -0.1:
        return 'negative'
    else:
        return 'neutral'

df['sentiment'] = df['clean_content'].apply(get_sentiment)
print(df['sentiment'].value_counts())

sentiment
neutral     43
positive    43
negative    14
Name: count, dtype: int64


In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_content'], df['sentiment'], test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 80
Test size: 20


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

vectorizer = TfidfVectorizer(max_features=200)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

print("Model trained successfully!")

Model trained successfully!


In [18]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.5

Classification Report:
               precision    recall  f1-score   support

    negative       0.00      0.00      0.00         4
     neutral       0.45      0.83      0.59         6
    positive       0.56      0.50      0.53        10

    accuracy                           0.50        20
   macro avg       0.34      0.44      0.37        20
weighted avg       0.41      0.50      0.44        20



c:\Users\Bisma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Bisma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Bisma\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [19]:
X_all_vec = vectorizer.transform(df['clean_content'])
df['predicted_sentiment'] = model.predict(X_all_vec)
df['confidence'] = model.predict_proba(X_all_vec).max(axis=1)

output = df[['content', 'predicted_sentiment', 'confidence']]
output.columns = ['post', 'sentiment', 'confidence']

print(output.head(10))

                                                post sentiment  confidence
0  “The world as we have created it is a process ...  positive    0.527090
1  “It is our choices, Harry, that show what we t...  positive    0.674686
2  “There are only two ways to live your life. On...   neutral    0.569096
3  “The person, be it gentleman or lady, who has ...   neutral    0.657558
4  “Imperfection is beauty, madness is genius and...  positive    0.482116
5  “Try not to become a man of success. Rather be...  positive    0.533541
6  “It is better to be hated for what you are tha...   neutral    0.527623
7  “I have not failed. I've just found 10,000 way...  positive    0.619308
8  “A woman is like a tea bag; you never know how...   neutral    0.535581
9  “A day without sunshine is like, you know, nig...   neutral    0.651167


In [21]:
output.to_csv('predictions.csv', index=False)
print("Saved!")

Saved!
